In [1]:

!pip -q install -U langchain langchain-community langchain-openai langchain-chroma langchain-text-splitters chromadb pypdf gradio


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


In [2]:

from langchain_community.document_loaders import PyPDFLoader

pdf_path = "/content/the_nestle_hr_policy_pdf_2012.pdf"

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print("Total PDF pages:", len(documents))
print(documents[0].page_content[:500])


/tmp/ipykernel_2832/794065335.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Total PDF pages: 8
Policy
Mandatory
September  2012
The Nestlé  
Human Resources Policy


In [3]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Total chunks created:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0].page_content[:500])


Total chunks created: 20

First chunk:

Policy
Mandatory
September  2012
The Nestlé  
Human Resources Policy


In [4]:
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")
print("API key loaded successfully!" if api_key else "API key missing")

API key loaded successfully!


In [6]:

!pip -q install sentence-transformers


In [7]:

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="nestle_hr_policy_free"
)

print("ChromaDB created successfully!")
print("Total chunks stored:", vector_db._collection.count())


/tmp/ipykernel_2832/2697902536.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ChromaDB created successfully!
Total chunks stored: 20


In [8]:

def retrieve_hr_policy(question):
    results = vector_db.similarity_search(question, k=3)

    context = "\n\n".join(
        doc.page_content for doc in results
    )

    return context

question = "What is Nestle's HR policy about?"

context = retrieve_hr_policy(question)

print("Relevant HR policy information:\n")
print(context[:1500])


Relevant HR policy information:

The Nestlé Human Resources Policy
1
At Nestlé, we recognize that our employees 
are the key to our success and nothing can be 
achieved without their engagement. 
This document encompasses the guidelines 
which constitute a solid basis for effective Human 
Resources Management throughout the Nestlé 
Group around the world. It explains to all Nestlé 
employees the vision and mission of the Human 
Resources function and illustrates every aspect of 
the Nestlé employee lifecycle. 
The Nestlé Management and Leadership 
Principles inspire all the Nestlé employees in their 
actions and in their dealings with others. The 
Corporate Business Principles refer to all the basic 
principles which Nestlé endorses and subscribes 
to on a worldwide basis. Both these documents 
are the pillars on which the present policy has 
been built.
The implementation of this policy will be 
inspired by sound judgement, compliance with 
local market laws and common sense, taking i

In [9]:

!pip -q install "transformers>=4.40,<5" sentencepiece

import gradio as gr
from transformers import pipeline

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    device=-1
)

def hr_chatbot(message, history):
    if not message.strip():
        return "Please enter an HR policy question."

    docs = vector_db.similarity_search(message, k=2)
    context = "\n\n".join(doc.page_content for doc in docs)

    prompt = f"""Answer the question using only the HR policy context.
If the answer is not present, say:
I could not find that information in the HR policy.

Context:
{context[:1800]}

Question: {message}
Answer:"""

    response = generator(
        prompt,
        max_new_tokens=150,
        do_sample=False,
        truncation=True
    )

    answer = response[0]["generated_text"].strip()

    if not answer:
        answer = "I could not find that information in the HR policy."

    pages = sorted(set(
        doc.metadata.get("page", 0) + 1
        for doc in docs
    ))

    return answer + "\n\nSource pages: " + ", ".join(map(str, pages))

demo = gr.ChatInterface(
    fn=hr_chatbot,
    title="Nestlé AI-Powered HR Assistant",
    description="Ask questions about Nestlé's Human Resources Policy.",
    examples=[
        "What is Nestlé's HR policy about?",
        "What are the key HR principles?",
        "What does the policy say about employees?"
    ]
)

demo.launch(share=True)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
Error importing huggingface_hub._login: cannot import name '_save_stored_tokens' from 'huggingface_hub.utils._auth' (/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py)


ImportError: cannot import name '_save_stored_tokens' from 'huggingface_hub.utils._auth' (/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py)

In [ ]:

%pip install -q --upgrade --force-reinstall "huggingface-hub==0.36.2" "transformers==4.57.6" "gradio==5.49.1" sentencepiece


In [1]:

import gradio as gr
from transformers import pipeline
import huggingface_hub

print("Gradio:", gr.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)
print("Imports successful!")


Gradio: 6.31.0
Hugging Face Hub: 0.36.2
Imports successful!


In [2]:

import gradio as gr
from transformers import pipeline

# Load the free AI model
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    device=-1
)

def hr_chatbot(message, history):
    if not message.strip():
        return "Please enter an HR policy question."

    # Find relevant information in the PDF
    docs = vector_db.similarity_search(message, k=2)

    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    # Create an AI prompt
    prompt = f"""Answer the question using only the context below.
If the answer is not available, say:
I could not find that information in the HR policy.

Context:
{context[:1500]}

Question: {message}
Answer:"""

    # Generate answer
    response = generator(
        prompt,
        max_new_tokens=150,
        do_sample=False,
        truncation=True
    )

    answer = response[0]["generated_text"].strip()

    if not answer:
        answer = "I could not find that information in the HR policy."

    pages = sorted(set(
        doc.metadata.get("page", 0) + 1
        for doc in docs
    ))

    return answer + "\n\nSource pages: " + ", ".join(map(str, pages))

# Build the chatbot interface
demo = gr.ChatInterface(
    fn=hr_chatbot,
    title="Nestlé AI-Powered HR Assistant",
    description="Ask questions about Nestlé's HR Policy.",
    examples=[
        "What is Nestlé's HR policy about?",
        "What are Nestlé's HR principles?",
        "What does the policy say about employees?"
    ]
)

demo.launch(share=True)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://40c6668a83faae1420.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
